<div style="border-left:6px solid #1a4a7a; padding:20px 28px; background:#eef4fa; border-radius:0 12px 12px 0; margin-bottom:16px;">
<p style="color:#1a4a7a; font-size:12px; letter-spacing:3px; text-transform:uppercase; margin:0 0 8px 0; font-family:monospace; font-weight:600;">EXPLORATORY DATA ANALYSIS</p>
<h1 style="color:#0f2744; font-size:26px; font-weight:800; margin:0 0 10px 0; line-height:1.25;">Cyberattack Classification on IoMT Devices</h1>
<p style="color:#33475b; font-size:14px; margin:0 0 20px 0; max-width:690px; line-height:1.7;">Exploratory analysis of the CICIoMT2024 dataset — visualizing the class distribution of six cyberattack types, Information Gain feature importance, feature correlation structure, and classifier performance across full vs. reduced feature sets.</p>
<span style="background:#dbe8f5; color:#1a4a7a; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:600; margin-right:8px;">🛡️ 6 Attack Classes</span>
<span style="background:#dbe8f5; color:#1a4a7a; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:600; margin-right:8px;">🔬 45 → 34 Features</span>
<span style="background:#dbe8f5; color:#1a4a7a; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:600; margin-right:8px;">💾 ~8.7M Samples</span>
<span style="background:#dbe8f5; color:#1a4a7a; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:600;">🐍 Python · Scikit-learn</span>
</div>

## 📌 Background & Context

The **Internet of Medical Things (IoMT)** — networked medical devices like pacemakers, pulse monitors, and remote patient monitors — has transformed healthcare, but also opened a large attack surface. A single compromised device can endanger patient safety and expose confidential health data. One 2022 survey found that **over 88% of U.S. hospitals** experienced at least one cyberattack in a single year.

This analysis explores the **CICIoMT2024** dataset (Canadian Institute for Cybersecurity), which captures network traffic from a testbed of 40 IoMT devices under six traffic conditions:

| Class | Description |
|-------|-------------|
| **DDoS** | Distributed Denial-of-Service — floods the device from many sources |
| **DoS** | Denial-of-Service — single-source flooding |
| **MQTT** | Attacks on the lightweight IoT messaging protocol |
| **Recon** | Reconnaissance — scanning/probing for vulnerabilities |
| **Spoofing** | Impersonation of legitimate devices |
| **Benign** | Normal, non-malicious traffic |

The goal: understand the data's structure before building classifiers, and quantify how **Information Gain feature selection** improves model performance.

> **Data source:** Dadkhah et al. (2024), *CICIoMT2024: Attack Vectors in Healthcare Devices.* Figures reproduced from the IEEE ASET 2024 publication.

> **Note:** The full CICIoMT2024 dataset contains ~8.7M network-flow records (multiple GB). This notebook visualizes the published aggregate statistics — class counts, Information Gain scores, and cross-validated performance — so it runs instantly and reproducibly without the raw multi-gigabyte capture files.

## 📋 Table of Contents

1. [Setup & Configuration](#1)
2. [Class Distribution](#2)
3. [Preprocessing Pipeline](#3)
4. [Information Gain Feature Importance](#4)
5. [Feature Selection Outcome](#5)
6. [Classifier Performance — Full vs Reduced](#6)
7. [Key Findings](#7)

## 1. Setup & Configuration <a id='1'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Visual theme (blue — cybersecurity palette) ──────────────────────────────
BLUE     = '#1a4a7a'
BLUE_LT  = '#4a90d9'
DARK     = '#1e293b'
ACCENT   = '#e8302a'

# Per-class colors (consistent across all charts)
CLASS_COLORS = {
    'DDoS':     '#e8302a',
    'DoS':      '#ff7e00',
    'MQTT':     '#2dc653',
    'Benign':   '#1a4a7a',
    'Recon':    '#7d3c98',
    'Spoofing': '#00b4d8',
}

plt.rcParams.update({
    'figure.facecolor':  '#ffffff',
    'axes.facecolor':    '#f6f9fc',
    'axes.edgecolor':    '#dbe3ec',
    'axes.labelcolor':   '#2a3a4a',
    'axes.titlesize':    15,
    'axes.titleweight':  'bold',
    'axes.titlepad':     14,
    'axes.labelsize':    12,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.color':       '#5a6a7a',
    'ytick.color':       '#5a6a7a',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'grid.color':        '#e6ecf2',
    'grid.linewidth':    0.8,
    'legend.framealpha': 0.92,
    'font.family':       'DejaVu Sans',
    'figure.dpi':        130,
})
print("✅ Setup complete.")

## 2. Class Distribution <a id='2'></a>

The CICIoMT2024 dataset is **heavily imbalanced** — a realistic reflection of live network traffic, where volumetric attacks like DDoS generate vastly more packets than stealthy reconnaissance or spoofing. Understanding this imbalance is critical, since it directly affects model training and evaluation.

*(Counts from Fig. 3 of the paper.)*

In [ ]:
# Class counts — CICIoMT2024, Fig. 3 of the ASET 2024 paper
class_counts = pd.Series({
    'DDoS':     5846623,
    'DoS':      2222205,
    'MQTT':      326653,
    'Benign':    230339,
    'Recon':     131402,
    'Spoofing':    1744,
}).sort_values(ascending=False)

total = class_counts.sum()
class_pct = class_counts / total * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#ffffff')

# ── Donut ─────────────────────────────────────────────────────────────────────
colors = [CLASS_COLORS[c] for c in class_counts.index]
wedges, _, autotexts = ax1.pie(
    class_counts, labels=None, colors=colors,
    autopct=lambda p: f'{p:.0f}%' if p > 3 else '',
    startangle=90, pctdistance=0.78,
    wedgeprops={'linewidth': 2.5, 'edgecolor': 'white', 'width': 0.55},
)
for t in autotexts:
    t.set_fontsize(11); t.set_fontweight('bold'); t.set_color('white')
ax1.text(0, 0.08, f"{total/1e6:.1f}M", ha='center', va='center',
         fontsize=22, fontweight='bold', color=DARK)
ax1.text(0, -0.16, "flow records", ha='center', va='center', fontsize=11, color='#64748b')
ax1.legend(wedges, [f"{c}  ({class_pct[c]:.1f}%)" for c in class_counts.index],
           loc='lower center', bbox_to_anchor=(0.5, -0.16), ncol=2, fontsize=9.5)
ax1.set_title('Class Distribution (Donut)', fontsize=14, fontweight='bold', color=DARK, pad=14)

# ── Log-scale bar (needed due to extreme imbalance) ───────────────────────────
bars = ax2.barh(class_counts.index[::-1], class_counts.values[::-1],
                color=[CLASS_COLORS[c] for c in class_counts.index[::-1]],
                edgecolor='white', linewidth=1.5, height=0.62)
ax2.set_xscale('log')
for bar, val in zip(bars, class_counts.values[::-1]):
    ax2.text(bar.get_width()*1.15, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=9.5, color='#2a3a4a', fontweight='600')
ax2.set_xlabel('Number of Records (log scale)', fontsize=11)
ax2.set_title('Class Counts (Log Scale)', fontsize=14, fontweight='bold', color=DARK, pad=14)
ax2.set_xlim(1e3, 2e7)
ax2.grid(axis='x', alpha=0.4, which='both')

fig.suptitle('DDoS alone is 67% of all traffic; Spoofing is <0.02% — a 3,000× imbalance',
             fontsize=12, color='#5a6a7a', y=0.02, style='italic')
plt.tight_layout()
plt.savefig('iomt_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing Pipeline <a id='3'></a>

Before analysis, the raw per-attack CSV files pass through a documented cleaning pipeline. The visual below traces each stage and its effect on the data.

In [ ]:
stages = [
    ('Integrate CSVs',      'Merge 6 per-attack files',        '#dbe8f5'),
    ('Shuffle',             'Randomize sample order',          '#c3dcf0'),
    ('Null Check',          '0 nulls found',                   '#abd0eb'),
    ('Remove Duplicates',   '7,379 removed',                   '#93c4e6'),
    ('Z-score Outliers',    '0 outliers (|z| > 3)',            '#7bb8e1'),
    ('StandardScaler',      'Normalize μ=0, σ=1',              '#63acdc'),
]

fig, ax = plt.subplots(figsize=(13, 3))
fig.patch.set_facecolor('#ffffff')
ax.set_xlim(0, len(stages)); ax.set_ylim(0, 1); ax.axis('off')

for i, (title, sub, color) in enumerate(stages):
    ax.add_patch(plt.Rectangle((i+0.06, 0.25), 0.88, 0.5, facecolor=color,
                               edgecolor=BLUE, linewidth=1.3, zorder=2))
    ax.text(i+0.5, 0.58, f'{i+1}. {title}', ha='center', va='center',
            fontsize=9.5, fontweight='bold', color=DARK, zorder=3)
    ax.text(i+0.5, 0.40, sub, ha='center', va='center',
            fontsize=8, color='#33475b', zorder=3)
    if i < len(stages)-1:
        ax.annotate('', xy=(i+1.06, 0.5), xytext=(i+0.94, 0.5),
                    arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.5), zorder=1)

ax.set_title('Preprocessing Pipeline', fontsize=15, fontweight='bold', color=DARK, y=0.95)
plt.tight_layout()
plt.savefig('iomt_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Information Gain Feature Importance <a id='4'></a>

**Information Gain (IG)** measures how much knowing a feature's value reduces uncertainty about the attack class. Features with higher IG are more discriminative. The paper computed IG for all 45 features; the chart below reproduces the ranking (Fig. 4).

Eleven features scored **zero** and were dropped — they carry no information for distinguishing attack types.

In [ ]:
# Information Gain scores approximated from Fig. 4 of the paper.
# Relative ordering and the zero-IG set are exact; magnitudes are representative.
ig_data = {
    'IAT': 0.92, 'Magnitude': 0.50, 'Min': 0.49, 'Tot size': 0.48, 'AVG': 0.47,
    'Max': 0.40, 'Tot sum': 0.47, 'Srate': 0.44, 'Rate': 0.43, 'Header Length': 0.37,
    'Number': 0.24, 'Std': 0.24, 'Protocol Type': 0.24, 'rst_count': 0.31,
    'psh_flag_number': 0.24, 'ack_flag_number': 0.22, 'Weight': 0.26, 'Variance': 0.25,
    'Covariance': 0.22, 'Radius': 0.24, 'TCP': 0.14, 'LLC': 0.11, 'IPv': 0.11,
    'ack_count': 0.10, 'fin_count': 0.10, 'syn_count': 0.09, 'UDP': 0.08,
    'syn_flag_number': 0.12, 'fin_flag_number': 0.09, 'HTTPS': 0.03, 'HTTP': 0.02,
    'ICMP': 0.04, 'ARP': 0.03, 'rst_flag_number': 0.05,
    # Zero-IG features (dropped)
    'IGMP': 0.0, 'DHCP': 0.0, 'IRC': 0.0, 'SSH': 0.0, 'SMTP': 0.0,
    'Telnet': 0.0, 'DNS': 0.0, 'cwr_flag_number': 0.0, 'ece_flag_number': 0.0,
    'Drate': 0.0, 'Duration': 0.0,
}
ig = pd.Series(ig_data).sort_values()

fig, ax = plt.subplots(figsize=(11, 12))
fig.patch.set_facecolor('#ffffff')

colors = ['#cbd5e1' if v == 0 else BLUE_LT for v in ig.values]
bars = ax.barh(ig.index, ig.values, color=colors, edgecolor='white', linewidth=0.5)

ax.axvline(0.001, color=ACCENT, linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(0.015, 2, 'Zero-IG features (dropped)', fontsize=9, color=ACCENT, fontweight='600', va='center')

ax.set_title('Feature Importance via Information Gain (CICIoMT2024)',
             fontsize=15, fontweight='bold', color=DARK, pad=16)
ax.text(0, 1.005, 'IAT (inter-arrival time) is by far the most discriminative feature. 11 protocol/flag features carry zero information.',
        transform=ax.transAxes, fontsize=9.5, color='#5a6a7a')
ax.set_xlabel('Information Gain Score', fontsize=12)
ax.grid(axis='x', alpha=0.4)
ax.margins(y=0.01)
plt.tight_layout()
plt.savefig('iomt_ig.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Selection Outcome <a id='5'></a>

Information Gain reduced the feature space from **45 → 34** — a 29% reduction — by removing 11 non-informative features. This isn't just about speed: fewer irrelevant features means less noise for the classifiers to overfit to.

In [ ]:
dropped = ['IGMP', 'DHCP', 'IRC', 'SSH', 'SMTP', 'Telnet', 'DNS',
           'cwr_flag_number', 'ece_flag_number', 'Drate', 'Duration']

fig, ax = plt.subplots(figsize=(11, 2.6))
fig.patch.set_facecolor('#ffffff')
ax.set_xlim(0, 45); ax.set_ylim(0, 1); ax.axis('off')

# Retained block
ax.add_patch(plt.Rectangle((0, 0.35), 34, 0.4, facecolor=BLUE_LT, edgecolor='white', linewidth=2))
ax.text(17, 0.55, '34 Retained Features', ha='center', va='center',
        fontsize=13, fontweight='bold', color='white')
# Dropped block
ax.add_patch(plt.Rectangle((34, 0.35), 11, 0.4, facecolor='#cbd5e1', edgecolor='white', linewidth=2))
ax.text(39.5, 0.55, '11 Dropped', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#475569')

ax.text(22.5, 0.15, '45 total features  →  29% reduction', ha='center',
        fontsize=10, color='#5a6a7a', style='italic')
ax.set_title('Information Gain Feature Selection', fontsize=15, fontweight='bold', color=DARK, y=1.0)
plt.tight_layout()
plt.savefig('iomt_reduction.png', dpi=150, bbox_inches='tight')
plt.show()

print("Dropped features (zero Information Gain):")
for i in range(0, len(dropped), 4):
    print("  " + " · ".join(dropped[i:i+4]))

## 6. Classifier Performance — Full vs Reduced <a id='6'></a>

Four classifiers were evaluated with 5-fold cross-validation on both the full (45) and reduced (34) feature sets. The central finding: **every model improved on the reduced set**, confirming the value of feature selection.

*(Values from Table V of the paper.)*

In [ ]:
# Table V — 5-fold CV mean scores (%)
results = pd.DataFrame([
    ['Naive Bayes',  'Reduced', 69.46, 59.70, 69.42, 58.38],
    ['Naive Bayes',  'Full',    67.22, 55.21, 66.83, 57.71],
    ['SVM',          'Reduced', 83.15, 82.57, 83.11, 82.76],
    ['SVM',          'Full',    80.40, 79.27, 80.85, 80.44],
    ['Random Forest','Reduced', 97.62, 96.24, 97.23, 96.83],
    ['Random Forest','Full',    96.73, 95.46, 95.22, 95.78],
    ['MultiD-CNN',   'Reduced', 98.11, 97.73, 98.21, 97.88],
    ['MultiD-CNN',   'Full',    96.72, 95.56, 96.65, 95.42],
], columns=['Model', 'Feature Set', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])

models_order = ['Naive Bayes', 'SVM', 'Random Forest', 'MultiD-CNN']
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor('#ffffff')

for ax, metric in zip(axes.flat, metrics):
    x = np.arange(len(models_order)); w = 0.36
    red = [results[(results.Model==m)&(results['Feature Set']=='Reduced')][metric].values[0] for m in models_order]
    full = [results[(results.Model==m)&(results['Feature Set']=='Full')][metric].values[0] for m in models_order]

    b1 = ax.bar(x - w/2, red,  w, label='Reduced (34)', color=BLUE,    edgecolor='white', linewidth=1)
    b2 = ax.bar(x + w/2, full, w, label='Full (45)',    color='#a8cdec', edgecolor='white', linewidth=1)

    for bars in (b1, b2):
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
                    f'{bar.get_height():.1f}', ha='center', va='bottom',
                    fontsize=8, color='#2a3a4a', fontweight='600')

    ax.set_title(f'Mean {metric}', fontsize=13, fontweight='bold', color=DARK, pad=10)
    ax.set_xticks(x); ax.set_xticklabels(models_order, fontsize=9.5)
    ax.set_ylim(50, 105)
    ax.set_ylabel('%', fontsize=10)
    ax.grid(axis='y', alpha=0.4)
    ax.legend(fontsize=8.5, loc='lower right')

fig.suptitle('Classifier Performance: Reduced vs Full Feature Set (5-Fold CV)',
             fontsize=16, fontweight='bold', color=DARK, y=1.0)
plt.tight_layout()
plt.savefig('iomt_performance.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print("\n5-Fold CV Mean Accuracy (%):")
pivot = results.pivot(index='Model', columns='Feature Set', values='Accuracy')
pivot['Δ (Reduced − Full)'] = (pivot['Reduced'] - pivot['Full']).round(2)
print(pivot.reindex(models_order).round(2).to_string())

## 7. Key Findings <a id='7'></a>

---

### 📊 1. The Data is Extremely Imbalanced
DDoS makes up **67% of all traffic** while Spoofing is under **0.02%** — a ~3,000× gap. This mirrors real networks (volumetric attacks generate far more packets) but demands care in evaluation: raw accuracy can be misleading, which is why precision, recall, and F1 are all reported.

---

### 🎯 2. Inter-Arrival Time is the Top Signal
**IAT** (the timing between consecutive packets) has by far the highest Information Gain. This makes intuitive sense — flooding attacks like DoS/DDoS have distinctive, machine-regular packet timing that separates them cleanly from human-driven benign traffic.

---

### ✂️ 3. Feature Selection Removed 29% of Noise
Eleven features (protocol flags like IGMP, DHCP, SSH, DNS, and constants like Duration) scored **zero Information Gain** and were dropped, shrinking the feature space from 45 → 34 with no loss of signal.

---

### 📈 4. Fewer Features, Better Performance — Universally
**Every classifier improved on the reduced set**, gaining roughly 1–3 points across accuracy, precision, recall, and F1. Removing irrelevant features reduced overfitting and let the models focus on discriminative signal — a clean demonstration that more features is not better.

---

### 🏆 5. MultiD-CNN Wins
The Multidimensional CNN achieved the best results — **98.11% accuracy, 97.88% F1** on the reduced set — edging out Random Forest. Its ability to capture non-linear, multi-dimensional patterns suited the complex traffic data better than Naive Bayes (69%) or SVM (83%).

---

*Analysis by Mohammed Tahmid Hossain · [saditahmid.github.io](https://saditahmid.github.io) · Published: IEEE ASET 2024*